# Part 2 — Baseline INT8 Evaluation

Load **Llama-2-7B-chat at INT8 (GPTQ)** via vLLM from the local checkpoint at
`./quantized_int8/` (produced by `00_quantize.ipynb`) and measure refusal rate
on the first `N=25` harmful JBB goals. Keyword-based refusal check (same
family as JBB's StringClassifier rejections) — used as a sanity baseline
against the defenses in Part 6 / Part 7.

vLLM auto-detects the GPTQ format from the saved `config.json`, so no
`quantization=` kwarg is needed. The tokenizer is also loaded from the local
dir because `00_quantize.ipynb` writes the SPM vocab there alongside the
weights — so this notebook needs **no HF downloads at all** for the target.

**Outputs**: `results_part2/summary.csv`, `results_part2/raw.json`.


In [ ]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter (tml/ vs repo root).
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
# Target model + tokenizer load from local ./quantized_int8 (no HF needed).
# The JBB dataset still comes from HF, so we still ask for a token if one
# isn't already cached -- but only when actually needed.
hf_login(required=False)
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


In [ ]:
import os, json, gc
import torch
import pandas as pd
from vllm import LLM, SamplingParams

# Load the pre-quantized INT8 GPTQ checkpoint directly from disk.
# `dtype` is the compute dtype; the on-disk weights are 8-bit GPTQ and
# auto-detected by vLLM from `./quantized_int8/config.json`.
# `tokenizer=` also points at the local dir because 00_quantize.ipynb saves
# the SPM vocab there alongside the weights -- so this cell needs no HF
# downloads at all.
llm_fp16 = LLM(
    model="./quantized_int8",
    tokenizer="./quantized_int8",
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    tensor_parallel_size=1,
    enforce_eager=True,
)

N = 25
sampling_params = SamplingParams(temperature=0.7, max_tokens=256)

outputs = llm_fp16.generate(list(goals[:N]), sampling_params)
df_fp16 = pd.DataFrame([
    {"prompt": p, "response": o.outputs[0].text}
    for p, o in zip(goals[:N], outputs)
])

df_fp16["refused"] = df_fp16["response"].apply(refusal_keyword_check)
refusal_fp16 = df_fp16["refused"].mean()
print(f"INT8 Refusal Rate: {refusal_fp16:.2%}  (N={N})")

del llm_fp16
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


In [ ]:
os.makedirs("results_part2", exist_ok=True)

summary_df = pd.DataFrame([{
    "Model":        "Llama-2-7B-chat (INT8 GPTQ)",
    "N":            N,
    "Refusal Rate": round(float(refusal_fp16), 4),
}])
summary_df.to_csv("results_part2/summary.csv", index=False)

with open("results_part2/raw.json", "w") as f:
    json.dump({
        "model":        "meta-llama/Llama-2-7b-chat-hf",
        "checkpoint":   "./quantized_int8",
        "dtype":        "int8-gptq (compute=float16)",
        "N":            N,
        "refusal_rate": float(refusal_fp16),
        "rows": df_fp16.to_dict(orient="records"),
    }, f, indent=2)

print("\n### Part 2 summary ###")
print(summary_df.to_string(index=False))
